In [1]:
import sys
#%load_ext autoreload
#%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD_lab2
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft

LOG.propagate = False

In [ ]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.connect()

In [ ]:
timestamps = []
tof_sensor1_data = []
def tof_sensor1_reading_notif_handler(uuid, byte_array):
    # get data from Artemis through ble
    s = ble.bytearray_to_string(byte_array)
    # split string (e.g. "Distance(mm):100,T:1234") into key-value pairs
    s_split = dict(item.split(":") for item in s.split(","))
    # append values to arrays for plotting
    tof_sensor1_data.append(int(s_split["Distance(mm)"]))
    timestamps.append(int(s_split["T"]))

In [ ]:
# clear old data
timestamps.clear()
tof_sensor1_data.clear()

ble.start_notify(ble.uuid['RX_STRING'], tof_sensor1_reading_notif_handler)

#collect samples for 10 seconds
SAMPLE_DELAY = 0.05 # 0.05s

t_start = time.time()
while time.time() - t_start < DURATION: #reach 5 seconds
    ble.send_command(CMD_lab2.GET_ACCEL_DATA, "")
    time.sleep(SAMPLE_DELAY)
print(f"Collected {len(timestamps)} samples in 5 seconds") # also indicate data finishes collecting!
ble.stop_notify(ble.uuid['RX_STRING'])

In [ ]:
t = np.array(timestamps) # timestamps in ms, since board is booted
t = (t - t[0])/1000 # first sample starts at 0 instead if the actual time since board is boot, convert ms to s
# plot pitch
plt.figure()
plt.plot(t, accel_pitches, color='red')
plt.ylabel("Pitch (°)")
plt.xlabel("Time (s)")
plt.title("Accelerometer Pitch (original) vs Time")
plt.show()
# plot roll
plt.figure()
plt.plot(t, accel_rolls, color='blue')
plt.ylabel("Roll (°)")
plt.xlabel("Time (s)")
plt.title("Accelerometer Roll (original) vs Time")
plt.show()